# 11a — Distributed data and associative aggregation — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/11.md) · [Course map](../PLAN.md) · [Project](../../projects/stage11/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Compute mergeable summaries
- Recognize shuffle and skew
- Choose analytical storage formats


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

Spark’s driver coordinates work; executors process partitions. Transformations build a lazy plan; actions trigger execution. A narrow transformation can operate locally; grouping and large joins often require a shuffle across machines. More partitions do not automatically improve performance because scheduling, network, and serialization have costs.

To average partitions correctly, merge sums and counts. Averaging partition means is wrong when partition sizes differ. Associative summaries allow distributed combination. Skew occurs when a few keys receive most records; one task can dominate runtime. Inspect partition sizes and physical plans before tuning.

CSV and JSON are broadly readable but weakly typed and expensive to scan. Parquet is columnar, typed, compressed, and suitable for analytical column selection. Avro is row-oriented with schema support and is often useful for record/event interchange. Choose a format based on access patterns, schema evolution, and interoperability. A laptop local-mode Spark lab teaches APIs, not cluster operations at scale.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
partitions = [[10, 20], [100]]
print("Wrong mean of means:", sum(sum(p)/len(p) for p in partitions)/len(partitions))
print("Correct:", sum(map(sum, partitions))/sum(map(len, partitions)))


## Exercise 1: Mergeable average

Return (sum,count) for each partition, combine, and return the global mean; reject zero total count.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def distributed_mean(partitions):
    raise NotImplementedError


In [ ]:
assert abs(distributed_mean([[10, 20], [100]])-130/3) < 1e-12
assert distributed_mean([[], [3]]) == 3
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Carry the denominator through aggregation.

</details>


## Exercise 2: Grouped partial sums

Merge a list of key→sum dictionaries into a new dictionary.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def merge_groups(partials):
    raise NotImplementedError


In [ ]:
assert merge_groups([{"a": 2}, {"a": 3, "b": 4}]) == {"a": 5, "b": 4}
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Addition is associative for exact integers.

</details>


## Exercise 3: Skew ratio

Return largest nonnegative partition size divided by mean size; reject empty or all-zero input.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def skew_ratio(sizes):
    raise NotImplementedError


In [ ]:
assert skew_ratio([10, 10, 10]) == 1
assert skew_ratio([0, 0, 30]) == 3
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Compare maximum to mean, then inspect the full distribution.

</details>


## Independent transfer

Generate a million synthetic transactions in batches, compute grouped totals without loading everything into memory, and compare output with Spark. Explain shuffle, broadcast joins, and skew.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Why is collect dangerous?

2. Why use an explicit schema?


## Reading and review

- [Primary reference 1](https://spark.apache.org/docs/latest/api/python/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
